# ⚙️ Panel Interactivo de Mantenimiento Predictivo — Maquinaria Rotativa

**Autor:** Enrique H.G. 

Este dashboard combina un dataset público de sensores industriales con un modelo de Machine Learning para **estimar en tiempo real la probabilidad de falla de una máquina** a partir de sus condiciones de operación (temperatura, velocidad rotacional, torque y desgaste de herramienta).

**Fuente de datos:** *AI4I 2020 Predictive Maintenance Dataset* — 10,000 registros sintéticos de un proceso de manufactura, con variables de proceso (temperatura de aire, temperatura de proceso, velocidad rotacional, torque, desgaste de herramienta) y el evento de falla asociado.

Este dataset está directamente alineado con problemas reales de **análisis de vibración, fatiga de herramienta y monitoreo de condición** en maquinaria rotativa — el mismo tipo de problema que aborda el mantenimiento predictivo industrial.

> Ejecutar las celdas en orden. El modelo se entrena **una sola vez** (Celda 6) y queda cacheado en memoria: mover los sliders del predictor **no** vuelve a entrenar nada, solo consulta el modelo ya ajustado.

In [1]:
# Importación de librerías y configuración del entorno
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

# Fuerza el renderizado de gráficos Plotly dentro de las celdas del notebook
pio.renderers.default = "notebook"

print("✅ Librerías cargadas correctamente. Entorno listo para Jupyter Notebook.")

✅ Librerías cargadas correctamente. Entorno listo para Jupyter Notebook.


In [2]:
# Descarga del dataset público (CSV vía URL) con manejo de errores

URL_DATASET = (
    "https://raw.githubusercontent.com/SamyamoyRakshit/"
    "AI4I-2020-Predictive-Maintenance-Dataset__Linear-Regression/"
    "main/ai4i2020.csv"
)

df_maquinaria = None
error_carga = None

try:
    df_maquinaria = pd.read_csv(URL_DATASET)
    if df_maquinaria.empty:
        raise ValueError("El archivo CSV se descargó vacío.")

    # Renombrado a nombres de columna manejables (sin corchetes ni unidades)
    df_maquinaria = df_maquinaria.rename(columns={
        "Air temperature [K]": "temp_aire_K",
        "Process temperature [K]": "temp_proceso_K",
        "Rotational speed [rpm]": "velocidad_rpm",
        "Torque [Nm]": "torque_Nm",
        "Tool wear [min]": "desgaste_min",
        "Machine failure": "falla",
        "Type": "tipo_producto",
    })

    print(f"✅ Dataset cargado: {df_maquinaria.shape[0]:,} registros, {df_maquinaria.shape[1]} columnas.")
    display(df_maquinaria.head())

except Exception as e:
    error_carga = f"{type(e).__name__}: {e}"
    display(Markdown(
        "### ⚠️ No fue posible descargar el dataset\n\n"
        "```\n" + error_carga + "\n```\n\n"
        "Verifica tu conexión a internet o que la URL siga disponible."
    ))

✅ Dataset cargado: 10,000 registros, 14 columnas.


,UDI,Product ID,tipo_producto,temp_aire_K,temp_proceso_K,velocidad_rpm,torque_Nm,desgaste_min,falla,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:
# Exploración interactiva de sensores (mini-dashboard EDA)

COLUMNAS_SENSOR = ["temp_aire_K", "temp_proceso_K", "velocidad_rpm", "torque_Nm", "desgaste_min"]

selector_variable = widgets.Dropdown(
    options=COLUMNAS_SENSOR,
    value="torque_Nm",
    description="Variable de sensor:",
    style={"description_width": "initial"},
)


def explorar_variable(variable):
    if df_maquinaria is None:
        display(Markdown("⚠️ El dataset no se cargó correctamente en la Celda 3."))
        return

    fig = px.histogram(
        df_maquinaria,
        x=variable,
        color=df_maquinaria["falla"].map({0: "Operación normal", 1: "Falla registrada"}),
        barmode="overlay",
        nbins=40,
        opacity=0.7,
        color_discrete_map={"Operación normal": "#2E86AB", "Falla registrada": "#D64550"},
        template="plotly_white",
        title=f"Distribución de '{variable}' — Operación normal vs. Falla",
    )
    fig.update_layout(height=420, legend_title="Estado de la máquina", bargap=0.02)
    fig.show()

    tasa_falla = df_maquinaria["falla"].mean() * 100
    print(f"Tasa de falla global en el dataset: {tasa_falla:.2f}%")


panel_eda = widgets.interactive(explorar_variable, variable=selector_variable)
display(Markdown("## 🔍 Exploración de sensores"))
display(panel_eda)

## 🔍 Exploración de sensores

interactive(children=(Dropdown(description='Variable de sensor:', index=3, options=('temp_aire_K', 'temp_proce…

In [4]:
# Preparación de variables para el modelo (sin fuga de información)

FEATURES_NUMERICAS = ["temp_aire_K", "temp_proceso_K", "velocidad_rpm", "torque_Nm", "desgaste_min"]
COLUMNA_TIPO = "tipo_producto"
COLUMNA_OBJETIVO = "falla"

# Se descartan las banderas de subtipo de falla (TWF, HDF, PWF, OSF, RNF):
# usarlas como predictoras sería fuga de información, ya que se calculan
# a partir del propio evento de falla.

codificador_tipo = LabelEncoder()
df_maquinaria["tipo_producto_cod"] = codificador_tipo.fit_transform(df_maquinaria[COLUMNA_TIPO])

LISTA_FEATURES = FEATURES_NUMERICAS + ["tipo_producto_cod"]

X = df_maquinaria[LISTA_FEATURES]
y = df_maquinaria[COLUMNA_OBJETIVO]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"✅ Variables preparadas: {len(LISTA_FEATURES)} features -> {LISTA_FEATURES}")
print(f"Entrenamiento: {X_train.shape[0]:,} registros | Prueba: {X_test.shape[0]:,} registros")

✅ Variables preparadas: 6 features -> ['temp_aire_K', 'temp_proceso_K', 'velocidad_rpm', 'torque_Nm', 'desgaste_min', 'tipo_producto_cod']
Entrenamiento: 7,500 registros | Prueba: 2,500 registros


In [5]:
# Entrenamiento del modelo (UNA sola vez) — se cachea en memoria

MODELO_FALLAS = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=3,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
MODELO_FALLAS.fit(X_train, y_train)

pred_test = MODELO_FALLAS.predict(X_test)
proba_test = MODELO_FALLAS.predict_proba(X_test)[:, 1]

exactitud = accuracy_score(y_test, pred_test)
auc = roc_auc_score(y_test, proba_test)

print(f"✅ Modelo entrenado y cacheado en memoria (variable global MODELO_FALLAS)")
print(f"Exactitud (accuracy): {exactitud:.3f}")
print(f"AUC-ROC: {auc:.3f}")
print("\nReporte de clasificación:\n")
print(classification_report(y_test, pred_test, target_names=["Sin falla", "Falla"]))

# Importancia de variables
importancias = pd.DataFrame({
    "variable": LISTA_FEATURES,
    "importancia": MODELO_FALLAS.feature_importances_,
}).sort_values("importancia", ascending=True)

fig_importancia = go.Figure(go.Bar(
    x=importancias["importancia"], y=importancias["variable"],
    orientation="h", marker_color="#2E86AB",
))
fig_importancia.update_layout(
    title="Importancia de variables en la predicción de fallas",
    template="plotly_white", height=380,
    xaxis_title="Importancia relativa", yaxis_title="",
    margin=dict(t=60, b=40),
)
fig_importancia.show()

✅ Modelo entrenado y cacheado en memoria (variable global MODELO_FALLAS)
Exactitud (accuracy): 0.975
AUC-ROC: 0.975

Reporte de clasificación:

              precision    recall  f1-score   support

   Sin falla       0.99      0.98      0.99      2415
       Falla       0.61      0.76      0.68        85

    accuracy                           0.98      2500
   macro avg       0.80      0.87      0.83      2500
weighted avg       0.98      0.98      0.98      2500



In [6]:
# Predictor en tiempo real — sliders conectados al modelo YA entrenado

rangos = df_maquinaria[FEATURES_NUMERICAS].describe()

slider_temp_aire = widgets.FloatSlider(
    value=float(df_maquinaria["temp_aire_K"].median()),
    min=float(rangos.loc["min", "temp_aire_K"]), max=float(rangos.loc["max", "temp_aire_K"]),
    step=0.1, description="Temp. aire (K):", style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"), readout_format=".1f",
)

slider_temp_proceso = widgets.FloatSlider(
    value=float(df_maquinaria["temp_proceso_K"].median()),
    min=float(rangos.loc["min", "temp_proceso_K"]), max=float(rangos.loc["max", "temp_proceso_K"]),
    step=0.1, description="Temp. proceso (K):", style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"), readout_format=".1f",
)

slider_velocidad = widgets.IntSlider(
    value=int(df_maquinaria["velocidad_rpm"].median()),
    min=int(rangos.loc["min", "velocidad_rpm"]), max=int(rangos.loc["max", "velocidad_rpm"]),
    step=10, description="Velocidad (rpm):", style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

slider_torque = widgets.FloatSlider(
    value=float(df_maquinaria["torque_Nm"].median()),
    min=float(rangos.loc["min", "torque_Nm"]), max=float(rangos.loc["max", "torque_Nm"]),
    step=0.5, description="Torque (Nm):", style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"), readout_format=".1f",
)

slider_desgaste = widgets.IntSlider(
    value=int(df_maquinaria["desgaste_min"].median()),
    min=int(rangos.loc["min", "desgaste_min"]), max=int(rangos.loc["max", "desgaste_min"]),
    step=1, description="Desgaste herramienta (min):", style={"description_width": "initial"},
    layout=widgets.Layout(width="420px"),
)

selector_tipo_pieza = widgets.Dropdown(
    options=list(zip(codificador_tipo.classes_, codificador_tipo.transform(codificador_tipo.classes_))),
    value=int(codificador_tipo.transform(["M"])[0]) if "M" in codificador_tipo.classes_ else 0,
    description="Calidad de pieza:",
    style={"description_width": "initial"},
)


def predecir_falla(temp_aire, temp_proceso, velocidad, torque, desgaste, tipo_pieza_cod):
    # NOTA: aquí solo se CONSULTA el modelo cacheado (MODELO_FALLAS de la Celda 6).
    # No se vuelve a llamar a .fit() en ningún momento.
    entrada = pd.DataFrame([{
        "temp_aire_K": temp_aire,
        "temp_proceso_K": temp_proceso,
        "velocidad_rpm": velocidad,
        "torque_Nm": torque,
        "desgaste_min": desgaste,
        "tipo_producto_cod": tipo_pieza_cod,
    }])[LISTA_FEATURES]

    probabilidad_falla = MODELO_FALLAS.predict_proba(entrada)[0, 1] * 100

    if probabilidad_falla < 15:
        color_riesgo, etiqueta = "#2E7D32", "Riesgo bajo"
    elif probabilidad_falla < 40:
        color_riesgo, etiqueta = "#F9A825", "Riesgo moderado"
    else:
        color_riesgo, etiqueta = "#C62828", "Riesgo alto"

    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=probabilidad_falla,
        number={"suffix": "%"},
        title={"text": f"Probabilidad de falla — {etiqueta}"},
        gauge={
            "axis": {"range": [0, 100]},
            "bar": {"color": color_riesgo},
            "steps": [
                {"range": [0, 15], "color": "#E8F5E9"},
                {"range": [15, 40], "color": "#FFF8E1"},
                {"range": [40, 100], "color": "#FFEBEE"},
            ],
        },
    ))
    fig.update_layout(height=380, template="plotly_white", margin=dict(t=60, b=20))
    fig.show()


panel_predictor = widgets.interactive(
    predecir_falla,
    temp_aire=slider_temp_aire,
    temp_proceso=slider_temp_proceso,
    velocidad=slider_velocidad,
    torque=slider_torque,
    desgaste=slider_desgaste,
    tipo_pieza_cod=selector_tipo_pieza,
)

display(Markdown(
    "## 🎛️ Predictor en tiempo real\n"
    "Ajusta las condiciones de operación y observa cómo cambia la probabilidad de falla "
    "estimada por el modelo (sin volver a entrenar)."
))
display(panel_predictor)

## 🎛️ Predictor en tiempo real
Ajusta las condiciones de operación y observa cómo cambia la probabilidad de falla estimada por el modelo (sin volver a entrenar).

interactive(children=(FloatSlider(value=300.1, description='Temp. aire (K):', layout=Layout(width='420px'), ma…